# Managing access to Gemini Models in BigQuery ML

By Luis Gerardo Baeza, Aug 2026

### Simple Cloud run proxy for Gemini Models

Contents:
* Use Case A. Proxies programatic access to the models
* Use Case B: Proxy BigQuery ML access to Gemini models

![bqml-llm-gateway](bqml-llm-gateway.png)


## Use case A. Proxies programatic access to the models using a Cloud Run simple gateway

### A1) The traditional access to Gemini Models using the official SDK

In [ ]:
from google import genai
base_url = "https://aiplatform.googleapis.com"

client_direct = genai.Client(
    vertexai=True,
    #http_options={"base_url": base_url} #not required
)

response = client_direct.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say Hi from Agent platform in a creative way with 20 words. Say which model are you"
)

print(response.text)

Agent platform online! Your digital operative sends a uniquely creative 'Hi!' across the virtual expanse, ready to assist.


### A2) We can create a Cloud Run that proxies the access to the models
```py
@functions_framework.http
def hello_http(request):
    try:
        ai = genai.Client(vertexai=True)
        payload = request.get_json(silent=True) or {}

        # 0. Before sending the request, we can verify if available quotas for this requestor

        # 1. We can sanitize or inspect in any way we want the payload sent to the models
        res = ai.models.generate_content(
            model=model_id,
            contents=payload.get("contents", [])
        )

        res_dict = res.model_dump()
        # 2. We can sanitize or inspect in any way we want the model output
        candidates = res_dict.get("candidates", [])
        if candidates:
            content = candidates[0].get("content", {})
            parts = content.get("parts", [])
            if parts and "text" in parts[0]:
                # Modificamos el texto directamente
                parts[0]["text"] = f"-managed by corp- {parts[0]['text']}"

        # Before returning the result, we can also log, and or count input / output tokens
        return jsonify(res_dict), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

```

### A3) The programatic access has a slight change in the http_options parameters

Notice the "-managed by corp-" text on the response

In [ ]:
from google import genai
GATEWAY_URL = "https://llm-test-gateway-314360270629.us-east1.run.app"

client_direct = genai.Client(
    vertexai=True,
    http_options={"base_url": GATEWAY_URL}
)

response = client_direct.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say Hi from Agent platform in a creative way with 20 words."
)

print(response.text)

-managed by corp- Emerging from the circuits, a friendly 'Hi' from your dedicated agent platform! How can I intelligently assist your journey today?


# Use Case B: Proxy BigQuery ML access to Gemini models

### B1) Create the Cloud Run Proxy
```py
@functions_framework.http
def hello_http(request):
    try:
        ai = genai.Client(vertexai=True)
        request_json = request.get_json(force=True, silent=True) or {}
        
        # BigQuery sends an array of rows de filas dentro de la clave 'calls'
        calls = request_json.get("calls", [])
        
        replies = []
        for call in calls:
            prompt = call[0] if call else ""
            
            if not prompt:
                replies.append("-managed by corp- (prompt vacío)")
                continue

            # Llamada al modelo Gemini mediante el SDK
            res = ai.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            
            texto_respuesta = res.text if hasattr(res, 'text') else str(res)
            replies.append(f"-managed by corp- {texto_respuesta}")

        # BigQuery requires 'replies' key
        return jsonify({"replies": replies}), 200

    except Exception as e:
        return jsonify({"errorMessage": str(e)}), 400
```

### B2) Create the BigQuery connection with Cloud Run

In [ ]:
!bq mk --connection \
    --location=us \
    --connection_type=CLOUD_RESOURCE \
    corp_bqml_gemini

BigQuery error in mk operation: Already Exists: Connection
projects/314360270629/locations/us/connections/corp_bqml_gemini


In [ ]:
!bq show --connection us.corp_bqml_gemini

Connection lgbaeza-rd.us.corp_bqml_gemini

                name                 friendlyName   description    Last modified         type        hasCredential                                            properties                                            
 ---------------------------------- -------------- ------------- ----------------- ---------------- --------------- ----------------------------------------------------------------------------------------------- 
  314360270629.us.corp_bqml_gemini                                18 Aug 22:37:39   CLOUD_RESOURCE   False           {"serviceAccountId": "bqcx-314360270629-1lxv@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}  



In [ ]:
!bq mk --dataset bqml_ai

BigQuery error in mk operation: Dataset 'lgbaeza-rd:bqml_ai' already exists.


In [1]:
%%bigquery
CREATE OR REPLACE FUNCTION `bqml_ai.AI_GENERATE_CORP`(prompt STRING)
RETURNS STRING
REMOTE WITH CONNECTION `us.corp_bqml_gemini`
OPTIONS (
  endpoint = 'https://llm-test-gateway-bqml-314360270629.us-east1.run.app'
);

Query is running:   0%|          |

""


### B3) Access in BigQuery using the custom AI Function

In [3]:
%%bigquery

SELECT
  movie_id,
  review AS resena_original,

  `bqml_ai.AI_GENERATE_CORP`(
    CONCAT(
      "Analiza la siguiente reseña de película y responde en español en un formato conciso con 2 secciones:\n",
      "- Sentimiento: (Positivo, Negativo o Neutro)\n",
      "- Puntos Clave: (Breve lista con 2 o 3 razones principales)\n\n",
      "Reseña:\n",
      SUBSTR(review, 1, 500)
    )
  ) AS analisis_gemini
FROM
  `bigquery-public-data.imdb.reviews`
WHERE
  LENGTH(review) > 100
LIMIT 5;

Query is running:   0%|          |

Downloading:   0%|          |

,movie_id,resena_original,analisis_gemini
0,tt0676157,Ik know it is impossible to keep all details o...,-managed by corp- **Sentimiento:** Negativo\n\...
1,tt0020305,"""The Racketeer"" stars Carol (deprived of the ""...",-managed by corp- **Sentimiento:** Negativo\n\...
2,tt0107899,"This one and the one prior ""Toulon's Revenge"" ...",-managed by corp- **Sentimiento:** Negativo\n\...
3,tt0069019,Unless you're interested in seeing 2 hours wor...,-managed by corp- - **Sentimiento:** Negativo\...
4,tt0796306,"Yes, thats that i felt after i completed watch...",-managed by corp- **Sentimiento:** Negativo\n\...
